# Raw data check — evidence for `documents/plan.md` §2

Each section below states one finding about the raw rainfall CSV, and then runs one Spark SQL query that shows it.
The expected output is what the 2017 file gave when we profiled it. If a different year gives different numbers, update the plan.

**How to run on Colab:** set `YEAR` below and run all cells. The CSV is copied from `gs://<DSA5208_GS_BUCKET>/raw/` (the same Colab secret as `download_dataset.ipynb`). Running locally, it reads `data/raw/` instead.

In [ ]:
!pip -q install pyspark

In [ ]:
import os

YEAR = 2017
FILE = f"rainfall_across_sg_{YEAR}.csv"

try:
    from google.colab import auth, userdata
    auth.authenticate_user()
    BUCKET = userdata.get("DSA5208_GS_BUCKET")   # e.g. gs://my-bucket
    LOCAL = f"/content/{FILE}"
    if not os.path.exists(LOCAL):
        os.system(f"gsutil cp {BUCKET}/raw/{FILE} {LOCAL}")
except ImportError:
    LOCAL = f"data/raw/{FILE}"

print(LOCAL, os.path.getsize(LOCAL) // 2**20, "MB")

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("raw-data-check")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.session.timeZone", "Asia/Singapore")
         .getOrCreate())

# Read every column as a string, so we see the raw text exactly as published.
raw = spark.read.csv(LOCAL, header=True, inferSchema=False)
raw.createOrReplaceTempView("raw")

# Same rows with parsed timestamps and the 5-minute bucket (timestamp rounded UP to the next 5-minute mark).
spark.sql("""
    SELECT *,
           to_timestamp(timestamp)                                   AS ts,
           to_timestamp(reading_update_timestamp)                    AS reading_upd,
           to_timestamp(update_timestamp)                            AS upd,
           timestamp_seconds(ceil(unix_seconds(to_timestamp(timestamp)) / 300) * 300) AS bucket,
           CAST(reading_value AS DOUBLE)                             AS v
    FROM raw
""").cache().createOrReplaceTempView("r")

def q(sql, n=30):
    spark.sql(sql).show(n, truncate=False)

raw.printSchema()

## 1. Size and empty cells

**Finding:** 5,256,106 rows, and no column has an empty cell. A missing reading is a missing row, not an empty value.

In [ ]:
cols = raw.columns
q("SELECT count(*) AS rows, " + ", ".join(f"sum(CASE WHEN `{c}` IS NULL OR `{c}` = '' THEN 1 ELSE 0 END) AS `{c}`" for c in cols) + " FROM raw")

## 2. Time zone and `date`

**Finding:** every timestamp ends in `+08:00` (Singapore time), and `date` always equals the date part of `timestamp`.

Expected: one row, `+08:00`, 5,256,106; `date_mismatch` = 0.

In [ ]:
q("SELECT substr(timestamp, 20) AS tz_offset, count(*) AS rows FROM raw GROUP BY 1")
q("SELECT sum(CASE WHEN date <> substr(timestamp, 1, 10) THEN 1 ELSE 0 END) AS date_mismatch FROM raw")

## 3. `timestamp` changes format in April 2017

**Finding:** until 2017-04-25 the seconds are `:59` (e.g. `08:04:59`); after that they are `:00` (e.g. `11:20:00`).

Expected: `:59` only in Jan–Mar, both in Apr, `:00` only from May. The `:59` rows end at `2017-04-25T11:24:59` and the `:00` rows start at `2017-04-25T11:20:00`, so they overlap.

In [ ]:
q("""
    SELECT substr(timestamp, 1, 7) AS month,
           sum(CASE WHEN substr(timestamp, 18, 2) = '59' THEN 1 ELSE 0 END) AS sec_59,
           sum(CASE WHEN substr(timestamp, 18, 2) = '00' THEN 1 ELSE 0 END) AS sec_00,
           sum(CASE WHEN substr(timestamp, 18, 2) NOT IN ('59', '00') THEN 1 ELSE 0 END) AS other
    FROM raw GROUP BY 1 ORDER BY 1
""")
q("""
    SELECT substr(timestamp, 18, 2) AS seconds, min(timestamp) AS first, max(timestamp) AS last
    FROM raw GROUP BY 1
""")

**Finding:** in the overlap, the same reading is published twice, one second apart (`11:19:59` and `11:20:00`), with identical values. So `:00` is just `:59` plus one second, and **both mark the end of the 5-minute interval**.

Expected: 2 pairs (`11:19:59`/`11:20:00` and `11:24:59`/`11:25:00`), 43 stations each, and `same_value` = 43.

In [ ]:
q("""
    SELECT a.timestamp AS ts_59, b.timestamp AS ts_00,
           count(*) AS stations,
           sum(CASE WHEN a.reading_value = b.reading_value THEN 1 ELSE 0 END) AS same_value
    FROM r a JOIN r b
      ON a.station_id = b.station_id AND b.ts = a.ts + INTERVAL 1 SECOND
    GROUP BY 1, 2 ORDER BY 1
""")

**Finding:** deduplicating on the raw `timestamp` finds nothing, because the two labels differ by one second. Rounding up to the 5-minute bucket **first** exposes the duplicates. That is why the clean step rounds before it dedupes.

Expected: `dups_on_raw_timestamp` = 0, `dups_on_bucket` = 86 (2 slots × 43 stations).

In [ ]:
q("""
    SELECT count(*) - count(DISTINCT station_id, timestamp) AS dups_on_raw_timestamp,
           count(*) - count(DISTINCT station_id, bucket)    AS dups_on_bucket
    FROM r
""")

## 4. The two update columns are not the same

**Finding:** `update_timestamp` and `reading_update_timestamp` differ in 932,997 rows (about 18%). When they differ, `update_timestamp` is always the later one. So `reading_update_timestamp` is when the reading changed, and it is the one to use for "keep the newest".

Expected: `equal` 4,323,109; `upd_later` 932,997; `upd_earlier` 0.

In [ ]:
q("""
    SELECT sum(CASE WHEN upd = reading_upd THEN 1 ELSE 0 END) AS equal,
           sum(CASE WHEN upd > reading_upd THEN 1 ELSE 0 END) AS upd_later,
           sum(CASE WHEN upd < reading_upd THEN 1 ELSE 0 END) AS upd_earlier
    FROM r
""")

How long after the reading is it updated? Expected: most within 15 minutes, a small tail over a day.

In [ ]:
q("""
    SELECT CASE WHEN lag_s < 0     THEN '0. negative'
                WHEN lag_s < 900   THEN '1. < 15 min'
                WHEN lag_s < 3600  THEN '2. < 1 h'
                WHEN lag_s < 86400 THEN '3. < 1 day'
                ELSE                    '4. >= 1 day' END AS reading_update_lag,
           count(*) AS rows
    FROM (SELECT unix_seconds(reading_upd) - unix_seconds(ts) AS lag_s FROM r)
    GROUP BY 1 ORDER BY 1
""")

## 5. Station identity

**Finding:** `station_name` is not unique: `S24` and `S24B` are both "Upper Changi Road North". `station_device_id` always equals `station_id`. So key on `station_id` only.

Expected: one name with two IDs; `device_differs` = 0.

In [ ]:
q("""
    SELECT station_name, collect_set(station_id) AS station_ids
    FROM raw GROUP BY 1 HAVING count(DISTINCT station_id) > 1
""")
q("SELECT sum(CASE WHEN station_device_id <> station_id THEN 1 ELSE 0 END) AS device_differs FROM raw")

## 6. Station coordinates can change

**Finding:** `S113` (Marine Parade Road) has two positions in 2017, about 40 m apart; it switches in April.

Expected: two rows for `S113`, the first used Jan–Apr, the second Apr–Dec.

In [ ]:
q("""
    WITH moved AS (
        SELECT station_id FROM raw
        GROUP BY 1 HAVING count(DISTINCT location_latitude, location_longitude) > 1
    )
    SELECT station_id, location_latitude, location_longitude,
           min(timestamp) AS first, max(timestamp) AS last, count(*) AS rows
    FROM raw WHERE station_id IN (SELECT station_id FROM moved)
    GROUP BY 1, 2, 3 ORDER BY 1, 4
""")
q("""
    SELECT station_id,
           round(111320 * sqrt(pow(max(lat) - min(lat), 2)
                             + pow((max(lon) - min(lon)) * cos(radians(avg(lat))), 2))) AS approx_distance_m
    FROM (SELECT station_id, CAST(location_latitude AS DOUBLE) AS lat, CAST(location_longitude AS DOUBLE) AS lon FROM raw)
    GROUP BY 1 HAVING count(DISTINCT lat, lon) > 1
""")

## 7. `reading_value`

**Finding:** values are in 0.2 mm steps (a tipping-bucket gauge); none are negative; 97% are zero; the maximum is 18.0 mm. A few carry float noise (e.g. `8.39999`), so values are snapped with `round(v * 5) / 5` before any threshold comparison.

Expected: `negative` 0, `max` 18.0, `zero_share` ≈ 0.972, `off_grid` 137.

In [ ]:
q("""
    SELECT count(*) AS rows,
           sum(CASE WHEN v < 0 THEN 1 ELSE 0 END)                   AS negative,
           max(v)                                                    AS max,
           round(avg(CASE WHEN v = 0 THEN 1.0 ELSE 0.0 END), 4)     AS zero_share,
           sum(CASE WHEN abs(v * 5 - round(v * 5)) > 1e-6 THEN 1 ELSE 0 END) AS off_grid
    FROM r
""")
q("""
    SELECT reading_value, round(v * 5) / 5 AS snapped, count(*) AS rows
    FROM r WHERE abs(v * 5 - round(v * 5)) > 1e-6
    GROUP BY 1, 2 ORDER BY 1
""")

## 8. Constant columns

**Finding:** `reading_type` and `reading_unit` each have a single value all year.

Expected: one row: `TB1 Rainfall 5 Minute Total F`, `mm`.

In [ ]:
q("SELECT reading_type, reading_unit, count(*) AS rows FROM raw GROUP BY 1, 2")

## 9. Stations start and stop during the year

**Finding:** 62 stations appear in 2017, but not all for the whole year. Some have only a handful of rows (`S97`: 6, `S72`: 33, `S103`: 45); some stop early (`S06`, `S110`, `S96`, `S24B`); some start late (`S111` in August, `S82` in November). So neighbour lists are built per year, with a minimum-coverage rule.

`coverage` = the station's rows ÷ the number of distinct 5-minute buckets in the year. Sorted from lowest coverage.

In [ ]:
q("""
    WITH d AS (SELECT DISTINCT station_id, bucket FROM r),
         n AS (SELECT count(DISTINCT bucket) AS buckets FROM r)
    SELECT station_id, count(*) AS rows,
           round(count(*) / max(n.buckets), 3) AS coverage,
           to_date(min(bucket)) AS first_day, to_date(max(bucket)) AS last_day
    FROM d CROSS JOIN n
    GROUP BY 1 ORDER BY coverage
""", n=70)

## 10. The whole network also goes silent

**Finding:** there are periods where **no** station reports. Expected: the year starts at `2017-01-01 08:05` (nothing before 08:05), about 273 gaps longer than 10 minutes, and the longest is a full day (2017-04-26 11:00 to 2017-04-27 11:40).

(Uses the rounded bucket, so the one-second duplicates from §3 don't show up as gaps.)

In [ ]:
q("SELECT min(bucket) AS first_bucket, max(bucket) AS last_bucket, count(DISTINCT bucket) AS buckets FROM r")
spark.sql("""
    SELECT prev, bucket AS next, (unix_seconds(bucket) - unix_seconds(prev)) / 60 AS gap_min
    FROM (SELECT bucket, lag(bucket) OVER (ORDER BY bucket) AS prev
          FROM (SELECT DISTINCT bucket FROM r))
    WHERE unix_seconds(bucket) - unix_seconds(prev) > 600
""").createOrReplaceTempView("gaps")
q("SELECT count(*) AS gaps_over_10_min FROM gaps")
q("SELECT * FROM gaps ORDER BY gap_min DESC", n=10)

**Finding:** even when the network is up, sometimes only a few stations report. Expected: 76 buckets with a single station, and 172 days with at least one bucket under 35 stations. This is why island features need a "how many reported" count.

In [ ]:
spark.sql("""
    SELECT bucket, count(DISTINCT station_id) AS stations FROM r GROUP BY 1
""").createOrReplaceTempView("per_bucket")
q("""
    SELECT CASE WHEN stations = 1  THEN '1. one station'
                WHEN stations < 10 THEN '2. 2-9'
                WHEN stations < 35 THEN '3. 10-34'
                ELSE                    '4. 35+' END AS stations_reporting,
           count(*) AS buckets
    FROM per_bucket GROUP BY 1 ORDER BY 1
""")
q("SELECT count(DISTINCT to_date(bucket)) AS days_with_a_bucket_under_35 FROM per_bucket WHERE stations < 35")